[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Python from the Start](https://johnfisher-ai.github.io/Python-Visual-Guides/python-from-the-start.html)

# Debugging


## What you will be able to do

Find out why code is doing something you did not expect, using a method rather than guesswork.
You will be able to inspect what a value actually contains, narrow a problem down to one line,
and recognize the kind of fix that hides a bug instead of solving it.


## The idea

### The problem

Every notebook so far has ended with errors, and each one came with a message naming the
problem. Real debugging starts where that stops.

Sometimes there is no error at all: the code runs, produces a number, and the number is wrong.
Sometimes the error is real but points at a line that is fine, because the actual mistake
happened earlier and only showed up there. Sometimes the code works on your data and fails on
someone else's, and you cannot see any difference between them.

In all three cases the instinct is to stare at the code and think harder. That is the slowest
available method, and for anything beyond a few lines it does not work.

### What debugging is

> **Debugging** is finding the difference between what you believe the code does and what it
> actually does. It is not a matter of intelligence or of reading carefully enough. It is a
> search, and it is done by checking beliefs one at a time until one of them turns out to be
> false.

That framing is the useful part. You are not looking for a mistake; you are looking for the
place where your model of the program and the program itself stop agreeing. Every technique in
this notebook is a way of asking the program what it is really doing, rather than assuming.

### The method

Four steps, in order. Skipping them is what turns twenty minutes into an afternoon.

1. **Reproduce it.** Find the smallest input that shows the problem every time. A bug you can
   only sometimes trigger cannot be investigated.
2. **Read the error you already have.** The **Errors and Exceptions** notebook covered this, and
   it is still the step people skip. The message and the last frame usually name the problem.
3. **Check your assumptions, one at a time.** Not "this should be a list of numbers" but
   `print(f"{values=}")`. Most bugs are a value that is not the shape you thought.
4. **Narrow it down.** Cut the input in half, or the code in half, and see which half still
   fails. Repeat. Ten steps of this will find a problem in a thousand lines.

### Where you will meet this

Constantly, and more as your programs grow. It is also the skill that transfers furthest: the
method above works in any language, and on things that are not code at all.

The **Testing and Packaging** guide covers the other half of this, which is writing tests so
that a bug is caught the moment it appears rather than weeks later.

### What this notebook covers

- Printing well: the `=` form that saves typing and mistakes
- `repr`, for differences you cannot see
- Checking the type and shape of a value before believing anything else about it
- Narrowing a problem down to one input, and one line
- A bug worked start to finish
- `assert`, for stating what must be true
- `breakpoint()` and Colab's `%debug`, shown rather than run
- `logging`, and when it beats `print`
- Three errors, including a fix that made the answer wrong instead of loud

### A first look

Before any of the detail, here is the idea in a few lines. There is nothing to run yet: read it,
and read the output underneath.

```python
values = ["18", "21", "", "24"]

print(f"{values=}")
print(f"{len(values)=}")
print(f"{[type(v).__name__ for v in values]=}")
```

```
values=['18', '21', '', '24']
len(values)=4
[type(v).__name__ for v in values]=['str', 'str', 'str', 'str']
```

Three lines, and the empty string is now visible where reading the code would not have shown it.
That is the whole activity: stop assuming, ask.


## Setup

One import, used only in the section on logging.

**Run this cell before the rest of the notebook.**


In [1]:
import logging

print("Ready.")


Ready.


## Worked examples

### Print well, not often

Everyone debugs with `print`. Most people do it badly, writing `print(x)` and then losing track
of which `x` produced which line.

The `=` form inside an f-string prints the expression **and** its value. It came up in the
**Strings** notebook; this is what it is for.


In [2]:
total = 47
items = ["a", "b"]

print(f"{total=}")
print(f"{items=}")
print(f"{len(items)=}")


total=47
items=['a', 'b']
len(items)=2


No label to type, no label to get wrong, and the output says exactly which expression it came
from. Any expression works, not only a name.

That last point matters more than it looks: `print(f"{len(items)=}")` answers a question about
`items` without you having to work out what the answer should be.


### repr, for differences you cannot see

`print` shows text the way it is meant to be read, which is exactly wrong when you are asking
why two things that look identical are not equal.


In [3]:
a = "42"
b = "42 "

print("with print:", a, "|", b)
print("are they equal?", a == b)


with print: 42 | 42 
are they equal? False


In [4]:
print(f"{a!r}")
print(f"{b!r}")


'42'
'42 '


`!r` inside an f-string is `repr`, from the **Strings** notebook. The trailing space is invisible
in the first cell and obvious in the second.

Reach for `repr` whenever something "looks right" and is not. Trailing whitespace, a tab that
looks like spaces, an empty string, `"None"` the text against `None` the value: all invisible to
`print` and all obvious to `repr`.


### Check the type before anything else

A large share of bugs are a value that is not the kind of thing you assumed.


In [5]:
value = "10"

print(f"{value=}")
print(f"{type(value).__name__=}")
print(f"{value + 5 if isinstance(value, int) else 'cannot add: it is text'}")


value='10'
type(value).__name__='str'
cannot add: it is text


`type(x).__name__` gives the type as a readable word. For a collection, `len` is the other
question worth asking immediately.

The **Values and Variables** notebook introduced `type()`. It stops being trivia the first time
a number arrives from a file as text and every calculation after it is wrong.


### Narrowing down

When the input is a collection and something in it is bad, do not inspect all of it. Test each
item and let the failure name itself.


In [6]:
rows = ["18", "21", "", "24", "22"]

for i, v in enumerate(rows):
    try:
        int(v)
    except ValueError as e:
        print(f"index {i} failed: {v!r} -> {e}")


index 2 failed: '' -> invalid literal for int() with base 10: ''


One line of output, and it names both the position and the value. Compare that with reading five
strings and trying to spot the empty one.

The same idea applies to code rather than data. When a long function fails, comment out the
second half and see whether it still fails. If it does, the problem is in the first half, and
you have halved the search. Ten rounds of that narrows a thousand lines to one.


### A bug, start to finish

Here is a function that looks correct.


In [7]:
rows = ["18", "21", "", "24", "22"]

def average(values):
    total = 0
    for v in values:
        total += int(v)
    return total / len(values)

average(rows)


ValueError: invalid literal for int() with base 10: ''

**Step 1, read the error.** `invalid literal for int() with base 10: ''` names the type, the
operation and the offending value. The empty string is the problem, and `int` is where it was
found.

**Step 2, confirm it in the data** rather than assuming.


In [8]:
for i, v in enumerate(rows):
    print(f"index {i}: {v!r}  len={len(v)}")


index 0: '18'  len=2
index 1: '21'  len=2
index 2: ''  len=0
index 3: '24'  len=2
index 4: '22'  len=2


Index 2 holds `''`. That is now a fact rather than a theory.

**Step 3, fix it, and check what the fix changed.**


In [9]:
def average(values):
    numbers = [int(v) for v in values if v.strip()]
    if not numbers:
        return None
    return sum(numbers) / len(numbers)

print(average(rows))


21.25


`21.25`, from four usable values. Note the denominator: `len(numbers)`, not `len(values)`.
Getting that wrong is the subject of the last error in this notebook.


### assert, for stating what must be true

An `assert` says "this should always hold", and stops immediately when it does not.


In [10]:
def mean(values):
    assert len(values) > 0, "mean of an empty list is undefined"
    return sum(values) / len(values)

print(mean([1, 2, 3]))


2.0


In [11]:
print(mean([]))


AssertionError: mean of an empty list is undefined

Without the assert, this would have been a `ZeroDivisionError` further down, which says less
about what went wrong.

Two things to know. An assert is for a condition you believe is always true, not for checking
user input; a bad value from outside your program deserves a real exception with a real message,
as the **Errors and Exceptions** notebook showed. And assertions can be **switched off** entirely
when Python is run with `-O`, so nothing that matters for correctness should depend on one.


### breakpoint, and the Colab debugger

When printing is not enough, you can stop the program and look around.

```python
def average(values):
    total = 0
    for v in values:
        breakpoint()        # execution stops here and hands you a prompt
        total += int(v)
    return total / len(values)
```

At the prompt you can print any name, run any expression, and step forward a line at a time. The
commands worth knowing are `p name` to print, `n` for the next line, `c` to continue, and `q` to
quit.

Neither this nor the next block is run in this notebook, because both wait for input and would
stop the notebook from finishing.

Colab has a second form that is often more useful, because it works **after** the error rather
than requiring you to predict where it will happen. Run a cell that fails, then in a new cell:

```python
%debug
```

That drops you into the frame where the exception was raised, with every local variable still
available to inspect. It is the fastest way to answer "what was that value at the moment it
broke".


### logging, when print stops being enough

`print` is fine while you are working. It becomes a problem when the prints stay in, because
there is no way to turn them off without editing every one.

`logging` separates the message from the decision about whether to show it.


In [12]:
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s", force=True)
log = logging.getLogger("demo")

log.debug("only shown when the level is DEBUG")
log.info("shown at INFO and below")
log.warning("shown at WARNING and below")


INFO: shown at INFO and below


The `debug` line produced nothing because the level is set to `INFO`. Change one line and every
debug message appears, without touching any of them.


In [13]:
logging.basicConfig(level=logging.DEBUG, format="%(levelname)s: %(message)s", force=True)

log.debug("now this one appears too")
log.info("and so does this")


DEBUG: now this one appears too


INFO: and so does this


Use `print` while you are actively working on something. Use `logging` for anything that will
run unattended, or for messages you want to keep but not always see.


## Your turn

Six tasks. Write your answer in the cell under each and run it.

Try each one before you look at an answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/20-debugging-solutions.ipynb).

**1.** Create `count = 12` and `label = "items"`, then print both using the `=` form in a single
f-string each.


In [14]:
# your code here


**2.** Given `first = "yes"` and `second = "yes "`, print both with `repr` and print whether
they are equal. Say in a comment what the difference is.


In [15]:
# your code here


**3.** Given `mixed = [1, "2", 3.0, None]`, print each item with its type name, one per line.


In [16]:
# your code here


**4.** Given `values = ["5", "6", "seven", "8"]`, print the index and value of every item that
cannot be converted with `int`.


In [17]:
# your code here


**5.** Write `first_item(items)` that returns `items[0]`, with an `assert` that the list is not
empty and a message saying so. Show it working and failing.


In [18]:
# your code here


**6.** Using the `values` from task 4, compute the average of the items that **are** numbers,
and print how many were skipped. Be careful with the denominator.


In [19]:
# your code here


## Common errors

Each cell below is run on purpose so you can see the real message.

### The quiet one: a fix that made the answer wrong

This is the most important cell in the notebook, and it raises nothing.


In [20]:
rows = ["18", "21", "", "24", "22"]

def average(values):
    total = 0
    for v in values:
        try:
            total += int(v)
        except ValueError:
            pass
    return total / len(values)

print(average(rows))


17.0


`17.0`. No error, no warning, and the correct answer is `21.25`.

The `except ValueError: pass` stopped the crash, and that felt like a fix. It was not. The
empty string is still skipped, but `len(values)` still counts it, so the total of four numbers
is divided by five.

A crash is a demand for attention. Silencing it without changing the logic converts a loud
problem into a quiet wrong answer, which is far more expensive. Whenever you catch an
exception, ask what the calculation after it now means.


In [21]:
def average(values):
    numbers = [int(v) for v in values if v.strip()]
    print(f"  using {len(numbers)} of {len(values)} values")
    return sum(numbers) / len(numbers) if numbers else None

print(average(rows))


  using 4 of 5 values
21.25


### AssertionError with nothing to go on

An `assert` with no message tells you where, and not what.


In [22]:
def scale(values, factor):
    assert factor > 0
    return [v * factor for v in values]

scale([1, 2, 3], -1)


AssertionError: 

`AssertionError` and nothing else. The line is shown, so you can work out what failed, but the
message had one job and did not do it.

Always give an assert a message, and put the offending value in it:


In [23]:
def scale(values, factor):
    assert factor > 0, f"factor must be positive, got {factor}"
    return [v * factor for v in values]

scale([1, 2, 3], -1)


AssertionError: factor must be positive, got -1

### Debugging the wrong thing

Not an exception, and worth naming because it costs the most time.


In [24]:
readings = ["21", "22", "23"]

def to_numbers(values):
    return [int(v) for v in values]

def average(numbers):
    return sum(numbers) / len(numbers)

print(average(to_numbers(readings)))


22.0


That works. Now suppose the answer had been wrong, and you spent an hour reading `average`.

`average` is four visible characters of arithmetic and is almost certainly correct. `to_numbers`
touches the data, and data is where problems come from. Before reading either, one line settles
which half to look at:


In [25]:
numbers = to_numbers(readings)

print(f"{readings=}")
print(f"{numbers=}")
print(f"{len(readings)=} {len(numbers)=}")


readings=['21', '22', '23']
numbers=[21, 22, 23]
len(readings)=3 len(numbers)=3


If `numbers` looks right, the problem is in `average`. If it does not, `average` was never worth
reading. Two seconds of output decides where the next hour goes, which is the whole argument for
checking instead of thinking.


## Recap

- Debugging is finding where your model of the program and the program disagree, by checking
  beliefs one at a time.
- Reproduce it, read the error, check assumptions, narrow it down. In that order.
- `print(f"{value=}")` prints the expression and its value, and works for any expression.
- `repr`, or `!r` in an f-string, shows differences that `print` hides.
- Check `type()` and `len()` before believing anything else about a value.
- Test each item and let the failure name itself, rather than reading the whole collection.
- `assert condition, "message"` states what must be true, and always deserves a message.
- Assertions can be disabled with `-O`, so never rely on one for correctness.
- `breakpoint()` stops the program; `%debug` in Colab inspects the frame after an error.
- Catching an exception without changing the logic turns a crash into a wrong answer.


## What is next

The **Where Next** notebook, the last in this guide. It maps what you now know onto the rest of
the library, and says which guide to read for which kind of work.


---

&#8592; **Previous:** [Environments and pip](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/python-from-the-start/19-environments-and-pip.ipynb)  &nbsp;·&nbsp;  [Python from the Start Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/python-from-the-start.html)
